In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math
import warnings
import re
warnings.filterwarnings("ignore")

In [2]:
data = pd.read_csv('D:/FSH/Personal/Minute/Project/DB/sampled_data.csv',encoding = 'utf-8', engine='python')

0         0.0
1         2.0
2         0.0
3         1.0
4         2.0
         ... 
149995    2.0
149996    2.0
149997    1.0
149998    1.0
149999    2.0
Name: rooms_count, Length: 150000, dtype: float64

<h2>Cleaning Function : Needs to be updated ... </h2>





In [13]:

#extract meter from descritpion
def extract_meter_numbers(description):
    if pd.isna(description):
        return np.nan
    matches = re.findall(r'(\d+)\s*متر\s*', description)
    if matches:
        return float(matches[0])
    return np.nan


#cleaning data frame function
def cleaning_data(data: pd.DataFrame) -> pd.DataFrame:

###->    # 1. Remove columns that more than 90% of its data is NA
    data = data.dropna(axis=1, thresh=int(0.1 * len(data)))

###->     #2. complete building size with 1 m^2 with number exctracter from description
    data['description'].astype(str)
    data['meter_numbers'] = data['description'].apply(extract_meter_numbers)
    counter = 0
    for row in data.itertuples():
      if row.building_size == 1:
        data['building_size'][counter] = row.meter_numbers
      counter += 1
    data = data.drop(columns=['meter_numbers'])

###->     #3. remove outliers from price and rent value by using price/M^2 and rent/M^2

    #calculate price per square meter and rent per square meter
    data['price_per_meter'] = (data['price_value'] / data['building_size'])/1000000

    #since we have credit and rent price we need total rent
    data['total_rent'] = data['rent_value']*30 + data['credit_value']
    data['rent_price_per_meter'] = (data['total_rent'] / data['building_size'])/1000000

    # Calculate Q1, Q3, and IQR for 'price_value'
    Q1_p = data['price_per_meter'].quantile(0.25)
    Q3_p = data['price_per_meter'].quantile(0.75)
    IQR_p = Q3_p - Q1_p

    # Define the lower and upper bounds for outliers in price values
    lower_bound_p = Q1_p - 1.5 * IQR_p
    upper_bound_p = Q3_p + 1.5 * IQR_p

    # Calculate Q1, Q3, and IQR for 'rent_value'
    Q1_r = data['rent_price_per_meter'].quantile(0.25)
    Q3_r = data['rent_price_per_meter'].quantile(0.75)
    IQR_r = Q3_r - Q1_r

    # Define the lower and upper bounds for outliers in rent values
    lower_bound_r = Q1_r - 1.5 * IQR_r
    upper_bound_r = Q3_r + 1.5 * IQR_r

    data = data[(data['price_per_meter'] > lower_bound_p) & (data['price_per_meter'] < upper_bound_p) |
                   (data['rent_price_per_meter'] > lower_bound_r) & (data['rent_price_per_meter'] < upper_bound_r)
                   ]
  

###->     #4. Make some columns' data standard

    ### rooms_count
    data['rooms_count'] = data['rooms_count'].replace({
   'یک' : 1,
    'دو' : 2,
    'سه' : 3,
    'چهار': 4,
    'بدون اتاق' : 0,
    'پنج یا بیشتر' : 5    })

    data['rooms_count'].astype('Int32')
    
    
    ### construction_year
    
    persian_to_english = str.maketrans('۰۱۲۳۴۵۶۷۸۹', '0123456789')

    df['construction_year'] = (
    df['construction_year']
    .astype(str)
    .replace('قبل از ۱۳۷۰', '1369')  
    .apply(lambda x: x.translate(persian_to_english))
    )

    df['construction_year'] = pd.to_numeric(df['construction_year'], errors = 'coerce').astype('Int64')

        

###->     #5. Add a Column called size_per_room
    
    data['size_per_room'] = data['building_size'] / (data['rooms_count'] + 1)
    
###->     #6. remove duplicated rows in data frame
    data = data.drop_duplicates(keep = 'last').reset_index(drop = True)

        
###->     #7.General Cleaning for boolian, numerical and object types of data
    
        
    numeric_features = [
            'rent_value', 'credit_value', 'rent_credit_transform',
            'transformable_price', 'transformable_credit', 'transformable_rent',
            'land_size', 'building_size', 'floor', 'rooms_count',
            'total_floors_count', 'unit_per_floor', 'construction_year',
            'location_latitude', 'location_longitude', 'location_radius', 
            'rent_price_per_meter', 'price_per_meter','total_rent'
        ]
    boolean_features = [
            'has_balcony', 'has_elevator', 'has_warehouse', 'has_parking',
            'is_rebuilt', 'has_warm_water_provider', 'has_heating_system',
            'has_cooling_system', 'has_restroom'
        ]
        
    categorical_features = [
            'cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
            'user_type', 'rent_type', 'price_mode', 'rent_mode',
            'deed_type', 'building_direction', 'floor_material'
        ]
        
    
    for col in numeric_features:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors='coerce').fillna(0)
        
    for col in boolean_features:
        if col in data.columns:
            data[col] = data[col].fillna(0)
        
    for col in categorical_features:
        if col in data.columns:   
            data[col] = data[col].fillna('unknown').astype(str)
    
    
###->     #8.Combine Price Mode and Rent Mode - Combine Price value and Rent value
    data['mode_rent_or_price'] = data['rent_mode'].combine_first(data['price_mode'])    
    data['value_rent_or_price'] = data['rent_price_per_meter'].combine_first(data['price_per_meter'])
    
###->     #9. Remove missing values in target column
    data = data.dropna(subset=['value_rent_or_price'])
    
    
    
   
    
    return data




df = cleaning_data(data)
df.shape

(125960, 47)

In [5]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'rent_mode',
       'rent_value', 'rent_type', 'price_mode', 'price_value', 'credit_mode',
       'credit_value', 'rent_credit_transform', 'transformable_price',
       'transformable_credit', 'transformable_rent', 'land_size',
       'building_size', 'deed_type', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_warm_water_provider', 'has_heating_system', 'has_cooling_system',
       'has_restroom', 'building_direction', 'floor_material',
       'location_latitude', 'location_longitude', 'location_radius',
       'price_per_meter', 'total_rent', 'rent_price_per_meter'],
      dtype='object')